In [ ]:
# SET UP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne
import os
import mne
import traceback
import gc
import matplotlib.pyplot as plt


In [ ]:
import os
import mne

# Set root directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
output_dir = os.path.join(root_dir, 'Group_Level_Results')
os.makedirs(output_dir, exist_ok=True)

event_id = {'Wd2N': 1, 'Wd2E': 2}  # Normal context = Wd2N, Slowed context = Wd2E

# Containers for evoked responses per condition
evokeds_E = []
evokeds_N = []

# Loop through each participant (filtering for IDs >= 10)
participants = sorted([
    p for p in os.listdir(root_dir)
    if os.path.isdir(os.path.join(root_dir, p)) and p.isdigit() and int(p) >= 10
])

for pid in participants:
    participant_path = os.path.join(root_dir, pid)
    epo_file = os.path.join(participant_path, f"{pid}_epo_clean.fif")

    if not os.path.exists(epo_file):
        print(f"[{pid}] ❌ Clean epochs not found.")
        continue

    try:
        epochs = mne.read_epochs(epo_file, preload=True)
        print(f"[{pid}] Loaded cleaned epochs.")

        # Subject-level averaging
        if 'Wd2E' in epochs.event_id:
            evoked_E = epochs['Wd2E'].average()
            evokeds_E.append(evoked_E)
        if 'Wd2N' in epochs.event_id:
            evoked_N = epochs['Wd2N'].average()
            evokeds_N.append(evoked_N)

    except Exception as e:
        print(f"[{pid}] ⚠️ Error loading or processing: {e}")
        continue

# GRAND AVERAGES
if evokeds_E:
    grand_avg_E = mne.grand_average(evokeds_E)
    grand_avg_E.save(os.path.join(output_dir, 'new_grand_average_E-ave.fif'))
    print("✅ Saved grand average ERP for Wd2E.")

if evokeds_N:
    grand_avg_N = mne.grand_average(evokeds_N)
    grand_avg_N.save(os.path.join(output_dir, 'new_grand_average_N-ave.fif'))
    print("✅ Saved grand average ERP for Wd2N.")


In [ ]:
import os
import mne
import matplotlib.pyplot as plt

# Load grand averages
grand_avg_E = mne.read_evokeds(os.path.join(output_dir, 'new_grand_average_E-ave.fif'), condition=0)
grand_avg_N = mne.read_evokeds(os.path.join(output_dir, 'new_grand_average_N-ave.fif'), condition=0)

# Define regions (as provided)
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

# Order for subplotting (row-major)
region_order = [
    'Left Anterior', 'Medial Anterior', 'Right Anterior',
    'Left Central', 'Medial Central', 'Right Central',
    'Left Posterior', 'Medial Posterior', 'Right Posterior'
]

# Create figure
fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.3, wspace=0.2)
colors = {'Normal Context': 'black', 'Slowed Context': 'gray'}

for i, region in enumerate(region_order):
    row, col = divmod(i, 3)
    ax = axes[row, col]

    picks = [ch for ch in regions[region] if ch in grand_avg_E.ch_names and ch in grand_avg_N.ch_names]
    if not picks:
        ax.set_visible(False)
        continue

    # Pick and average over region channels
    evoked_E_region = grand_avg_E.copy().pick_channels(picks)
    evoked_N_region = grand_avg_N.copy().pick_channels(picks)



    # Plot using MNE's compare function
    mne.viz.plot_compare_evokeds(
       {'Slowed Context': evoked_E_region, 'Normal Context': evoked_N_region},
       picks=picks,
       axes=ax,
       colors=colors,
       show=False,
       split_legend=False,
       ci=0.68,
       combine='mean'
       )

    # Shade N100: typically ~100–160 ms
    ax.axvspan(0.100, 0.160, color='black', alpha=0.1)

    # Shade N280: typically ~240–300 ms
    ax.axvspan(0.240, 0.300, color='black', alpha=0.1)

    ax.set_title(region.replace(" ", "\n"), fontsize=10)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('')
    ax.set_ylabel('')

# Shared x/y labels
fig.text(0.5, 0.04, 'Time (ms)', ha='center')
fig.text(0.06, 0.5, 'Amplitude (µV)', va='center', rotation='vertical')

# Custom legend
fig.legend(['Normal Context', 'Slowed Context'], loc='lower center', ncol=2, frameon=False)

# Save or show
plt.suptitle('Grand Average ERPs by Scalp Region')
plt.savefig(os.path.join(output_dir, 'group_ERP_topo_grid.png'), dpi=300)
plt.show()


In [ ]:
import os
import mne
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Load grand averages
grand_avg_E = mne.read_evokeds(os.path.join(output_dir, 'grand_average_E-ave.fif'), condition=0)
grand_avg_N = mne.read_evokeds(os.path.join(output_dir, 'grand_average_N-ave.fif'), condition=0)

# Define regions (as provided)
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

region_order = [
    'Left Anterior', 'Medial Anterior', 'Right Anterior',
    'Left Central', 'Medial Central', 'Right Central',
    'Left Posterior', 'Medial Posterior', 'Right Posterior'
]

# Create figure with extra width for legend
fig, axes = plt.subplots(3, 3, figsize=(14, 8), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.3, wspace=0.3, right=0.82)

styles = {
    'Normal Context': dict(color='black', linestyle='-'),
    'Slowed Context': dict(color='black', linestyle='--')
}

for i, region in enumerate(region_order):
    row, col = divmod(i, 3)
    ax = axes[row, col]

    picks = [ch for ch in regions[region] if ch in grand_avg_E.ch_names and ch in grand_avg_N.ch_names]
    if not picks:
        ax.set_visible(False)
        continue

    evoked_E_region = grand_avg_E.copy().pick_channels(picks)
    evoked_N_region = grand_avg_N.copy().pick_channels(picks)

    mne.viz.plot_compare_evokeds(
        {'Slowed Context': evoked_E_region, 'Normal Context': evoked_N_region},
        picks=picks,
        axes=ax,
        styles=styles,
        show=False,
        split_legend=False,
        ci=0.68,
        combine='mean'
    )

    # Shade N100 and N280
    ax.axvspan(0.100, 0.160, color='skyblue', alpha=0.3)
    ax.axvspan(0.240, 0.300, color='lightpink', alpha=0.3)

    ax.set_title(region.replace(" ", "\n"), fontsize=10)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('')
    ax.set_ylabel('')

# Shared axis labels
fig.text(0.5, 0.04, 'Time (ms)', ha='center')
fig.text(0.06, 0.5, 'Amplitude (µV)', va='center', rotation='vertical')

# Legend items
line_norm = mlines.Line2D([], [], color='black', linestyle='-', label='Normal Context')
line_slow = mlines.Line2D([], [], color='black', linestyle='--', label='Slowed Context')
shade_n100 = mpatches.Patch(color='skyblue', alpha=0.3, label='N100 window')
shade_n280 = mpatches.Patch(color='lightpink', alpha=0.3, label='N280 window')

fig.legend(handles=[line_norm, line_slow, shade_n100, shade_n280],
           loc='center right', bbox_to_anchor=(0.88, 0.5), frameon=False)

plt.suptitle('Grand Average ERPs by Scalp Region')
plt.savefig(os.path.join(output_dir, 'group_ERP_topo_grid.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import os
import mne
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Load grand averages
grand_avg_E = mne.read_evokeds(os.path.join(output_dir, 'new_grand_average_E-ave.fif'), condition=0)
grand_avg_N = mne.read_evokeds(os.path.join(output_dir, 'new_grand_average_N-ave.fif'), condition=0)

# Define regions
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

region_order = [
    'Left Anterior', 'Medial Anterior', 'Right Anterior',
    'Left Central', 'Medial Central', 'Right Central',
    'Left Posterior', 'Medial Posterior', 'Right Posterior'
]

# Create figure with extra bottom space
fig, axes = plt.subplots(3, 3, figsize=(14, 9), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.35, wspace=0.25, bottom=0.15)

styles = {
    'Normal Context': dict(color='black', linestyle='-'),
    'Slowed Context': dict(color='black', linestyle='--')
}

for i, region in enumerate(region_order):
    row, col = divmod(i, 3)
    ax = axes[row, col]

    picks = [ch for ch in regions[region] if ch in grand_avg_E.ch_names and ch in grand_avg_N.ch_names]
    if not picks:
        ax.set_visible(False)
        continue

    evoked_E_region = grand_avg_E.copy().pick_channels(picks)
    evoked_N_region = grand_avg_N.copy().pick_channels(picks)

    mne.viz.plot_compare_evokeds(
        {'Slowed Context': evoked_E_region, 'Normal Context': evoked_N_region},
        picks=picks,
        axes=ax,
        styles=styles,
        show=False,
        split_legend=False,
        ci=0.68,
        combine='mean'
    )

    # Shade component windows
    ax.axvspan(0.100, 0.160, color='skyblue', alpha=0.3)
    ax.axvspan(0.240, 0.300, color='lightpink', alpha=0.3)

    ax.set_title(region.replace(" ", "\n"), fontsize=10)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('')
    ax.set_ylabel('')

# Global axis labels
fig.text(0.5, 0.06, 'Time (ms)', ha='center')
fig.text(0.06, 0.5, 'Amplitude (µV)', va='center', rotation='vertical')

# Single bottom legend
line_norm = mlines.Line2D([], [], color='black', linestyle='-', label='Normal Context')
line_slow = mlines.Line2D([], [], color='black', linestyle='--', label='Slowed Context')
shade_n100 = mpatches.Patch(color='skyblue', alpha=0.3, label='N100 window')
shade_n280 = mpatches.Patch(color='lightpink', alpha=0.3, label='N280 window')

fig.legend(
    handles=[line_norm, line_slow, shade_n100, shade_n280],
    loc='lower center',
    bbox_to_anchor=(0.5, 0.01),
    ncol=4,
    frameon=False
)

# Save & show
plt.suptitle('Grand Average ERPs by Scalp Region', fontsize=14)
plt.savefig(os.path.join(output_dir, 'group_ERP_topo_grid_cleaned.png'), dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import os
import mne
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Load grand averages
grand_avg_E = mne.read_evokeds(os.path.join(output_dir, 'new_grand_average_E-ave.fif'), condition=0)
grand_avg_N = mne.read_evokeds(os.path.join(output_dir, 'new_grand_average_N-ave.fif'), condition=0)

# Define regions
regions = {
    'Left Anterior':     ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20'],
    'Medial Anterior':   ['E12', 'E5', 'E19', 'E11', 'E4', 'E18', 'E16', 'E10', 'E15', 'E21', 'E14', 'E17', 'E127', 'E126'],
    'Right Anterior':    ['E117', 'E118', 'E124', 'E110', 'E109', 'E116', 'E123', 'E3', 'E9', 'E2', 'E122', 'E115', 'E114', 'E121', 'E1', 'E8', 'E125', 'E120', 'E113', 'E119'],
    'Left Central':      ['E29', 'E30', 'E41', 'E45', 'E46', 'E42', 'E47', 'E49', 'E37', 'E36'],
    'Medial Central':    ['E6', 'E7', 'E13', 'E31', 'E106', 'E112', 'E80', 'E54', 'E55', 'E79'],
    'Right Central':     ['E87', 'E93', 'E98', 'E102', 'E103', 'E107', 'E108', 'E104', 'E105', 'E111'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E53', 'E56', 'E57', 'E58', 'E59', 'E60', 'E63', 'E64', 'E65', 'E66', 'E68', 'E69', 'E70', 'E73'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77', 'E71', 'E76', 'E75', 'E74', 'E78', 'E82', 'E81'],
    'Right Posterior':   ['E83', 'E84', 'E85', 'E86', 'E88', 'E89', 'E90', 'E91', 'E92', 'E94', 'E95', 'E96', 'E97', 'E99', 'E100', 'E101']
}

region_order = [
    'Left Anterior', 'Medial Anterior', 'Right Anterior',
    'Left Central', 'Medial Central', 'Right Central',
    'Left Posterior', 'Medial Posterior', 'Right Posterior'
]

# Create figure
fig, axes = plt.subplots(3, 3, figsize=(14, 9), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.35, wspace=0.25, bottom=0.18)

styles = {
    'Normal Context': dict(color='black', linestyle='-'),
    'Slowed Context': dict(color='black', linestyle='--')
}

# Loop through regions
for i, region in enumerate(region_order):
    row, col = divmod(i, 3)
    ax = axes[row, col]

    picks = [ch for ch in regions[region] if ch in grand_avg_E.ch_names and ch in grand_avg_N.ch_names]
    if not picks:
        ax.set_visible(False)
        continue

    evoked_E_region = grand_avg_E.copy().pick_channels(picks)
    evoked_N_region = grand_avg_N.copy().pick_channels(picks)

    # Plot without individual legends
    mne.viz.plot_compare_evokeds(
        {'Slowed Context': evoked_E_region, 'Normal Context': evoked_N_region},
        picks=picks,
        axes=ax,
        styles=styles,
        show=False,
        split_legend=False,
        ci=0.68,
        combine='mean'
    )

    # Remove automatic subplot legend
    if ax.get_legend():
        ax.get_legend().remove()

    # Shade N100 and N280 windows
    ax.axvspan(0.100, 0.160, color='skyblue', alpha=0.3)
    ax.axvspan(0.240, 0.300, color='lightpink', alpha=0.3)

    ax.set_title(region.replace(" ", "\n"), fontsize=10)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel('')
    ax.set_ylabel('')

# Shared axis labels
fig.text(0.5, 0.06, 'Time (ms)', ha='center')
fig.text(0.06, 0.5, 'Amplitude (µV)', va='center', rotation='vertical')

# Global legend at bottom
line_norm = mlines.Line2D([], [], color='black', linestyle='-', label='Normal Context')
line_slow = mlines.Line2D([], [], color='black', linestyle='--', label='Slowed Context')
shade_n100 = mpatches.Patch(color='skyblue', alpha=0.3, label='N100 window')
shade_n280 = mpatches.Patch(color='lightpink', alpha=0.3, label='N280 window')

fig.legend(
    handles=[line_norm, line_slow, shade_n100, shade_n280],
    loc='lower center',
    bbox_to_anchor=(0.5, 0.02),
    ncol=4,
    frameon=False
)

plt.suptitle('Grand Average ERPs by Scalp Region', fontsize=14)
plt.savefig(os.path.join(output_dir, '1_new_group_ERP_topo_grid_cleaned_final.png'), dpi=300, bbox_inches='tight')
plt.show()
